In [1]:
# === Imports (один раз в начале ноутбука) ===
import sys, torch
import sys, importlib
from snn_mnist_net import Cfg, run_experiment, save_snn, load_weights_into, build_net,build_encoder_from_cfg
from label_map import build_label_map,save_label_map, load_label_map
from counts_readout import collect_counts_plus_cuda, make_mnist_datasets
from readout_models import tfidf_from_counts, train_mlp_readout
# (опционально) from evaluation import evaluate_on_mnist

_MODS = [
    "snn_mnist_net",
    "label_map",
    "counts_readout",
    "readout_models",
]

# 1) вычистим из sys.modules сам модуль и подпакеты
for m in list(sys.modules):
    if m in _MODS or any(m.startswith(x + ".") for x in _MODS):
        sys.modules.pop(m, None)

# 2) инвалидируем кэши файловой системы
importlib.invalidate_caches()

# 3) заново импортируем модули как модули
import snn_mnist_net, label_map, counts_readout, readout_models

In [2]:


# === FULL RUN CONFIG (адаптировано под Cfg) ===
cfg = Cfg(
    # базовые
    time=200, n_hidden=100, device="cpu", seed=42,
    # обучение
    N=60000,                 # стартовый «полный» прогон
    log_every=500,
    # STDP
    nu_plus=1e-4, nu_minus=-1e-3,
    # ингибиция / WTA
    enable_inhibition_at_start=True,
    top_k=3,
    inhib_strength=0.705,
    # веса
    w_init_lo=0.25, w_init_hi=0.80,
    w_clip_min=0.0, w_clip_max=1.0,
    # пороги/динамика LIF
    thresh_init=0.38, tau_val=150.0, refrac_val=2.0, reset_val=0.0,
    # кодировщик
    encoder="latency", poisson_rate_scale=0.011,
    # прочее
    debug=False,
    warmup_N = 100,
    thresh_min = 0.05,
    bootstrap_threshold = 0.12
    
)
print("FULL RUN:", {k: getattr(cfg, k) for k in [
    "inhib_strength","w_init_lo","w_init_hi","poisson_rate_scale","thresh_init","N"
]})




FULL RUN: {'inhib_strength': 0.705, 'w_init_lo': 0.25, 'w_init_hi': 0.8, 'poisson_rate_scale': 0.011, 'thresh_init': 0.38, 'N': 60000}


In [3]:
import importlib, snn_mnist_net
importlib.reload(snn_mnist_net)
# === 1) Обучение SNN (STDP) ===
summary, FF, lif_layer, net, encoder = run_experiment(cfg, verbose=True, progress=True)
print("SUMMARY:", summary)

print("SUMMARY:", summary)
save_snn("out/snn_mnist_v6_latency.pt", cfg, FF, lif_layer)

Используем: cpu

SNN-MNIST TRAIN START
seed=42  device=cpu  T=200  N=60000
n_hidden=100  thresh_init=0.38
STDP: nu_plus=0.0001  nu_minus=-0.001
FF init: w∈[0.25, 0.8], clip=[0.0,1.0]
Encoder=latency  poisson_rate_scale=0.011  base_seed=123
Inhibition: enable=True  inhib_strength=0.705  top_k=3
---------------------------

[lif] thresh = tensor mean=0.380 std=0.000 shape=(100,)
[lif] tc_decay = scalar 150.000
[lif] refrac = scalar 2.000
[lif] reset = scalar 0.000
[lif] dt = scalar 1.000


Warmup (T=200):   1%|▏                  | 1/100 [00:00<00:16,  6.13it/s, avg_rate=0.000 target=1.50]

[enc] shape=(200, 1, 784) sum=784.0 max=1.0
[enc] per_t: min=0.0 max=618.0 t_argmax=199
[mon] in_sum=784.0 lif_sum=0.0 v_max=-20.665809631347656 vt_min=0.15000000596046448
[enc] shape=(200, 1, 784) sum=784.0 max=1.0
[enc] per_t: min=0.0 max=608.0 t_argmax=199


Warmup (T=200):   3%|▌                  | 3/100 [00:00<00:14,  6.83it/s, avg_rate=0.000 target=1.50]

[mon] in_sum=784.0 lif_sum=0.0 v_max=-64.88957214355469 vt_min=0.1498500108718872
[enc] shape=(200, 1, 784) sum=784.0 max=1.0
[enc] per_t: min=0.0 max=664.0 t_argmax=199
[mon] in_sum=784.0 lif_sum=0.0 v_max=-64.91765594482422 vt_min=0.14970001578330994


Warmup (T=200): 100%|█████████████████| 100/100 [00:13<00:00,  7.48it/s, avg_rate=0.000 target=1.50]
Train (N=60000, T=200):  44%|▍| 26559/60000 [1:08:52<1:26:43,  6.43it/s, spikes=0.00 uniq=0 HHI=0.00


KeyboardInterrupt: 

In [3]:
import os
import importlib
import evaluation 
from evaluation import probe_readouts_counts
importlib.invalidate_caches()
importlib.reload(snn_mnist_net)
importlib.reload(evaluation)
importlib.invalidate_caches()
net, input_layer, lif_layer, connection, recurrent_inh, _ = build_net(cfg)
enc = build_encoder_from_cfg(cfg)

# 2) загрузить веса и пороги
load_weights_into(net, connection, lif_layer, "out/snn_mnist_v5.pt")



[lif] thresh = tensor mean=0.380 std=0.000 shape=(100,)
[lif] tc_decay = scalar 150.000
[lif] refrac = scalar 2.000
[lif] reset = scalar 0.000
[lif] dt = scalar 1.000
Loaded from out/snn_mnist_v5.pt


In [4]:
# 3) теперь вызываем build_label_map
label_map_build = build_label_map(net, None, lif_layer, enc, n_calib=2000, T=cfg.time, top_k=3, seed=123)

save_label_map("out/label_map_v5.pt",label_map_build, meta={"ckpt":"out/snn_mnist_v5.pt","T":cfg.time})

KeyboardInterrupt: 

In [5]:
label_map_build = load_label_map("out/label_map_v5.pt")

[label_map] loaded: out/label_map_v5.pt  shape=(100,)  meta={'created_at': '2026-01-26 12:18:18', 'ckpt': 'out/snn_mnist_v5.pt', 'T': 200}


In [6]:
from counts_readout import make_mnist_datasets

ds_train, ds_test = make_mnist_datasets()

In [ ]:
import importlib

from counts_readout import collect_counts_plus_fast, make_mnist_datasets
from evaluation import probe_readouts_counts, train_mlp_readout
importlib.invalidate_caches()
importlib.reload(counts_readout)
Xtr, ytr, _ = collect_counts_plus_fast(
    net, lif_layer, enc, ds_train, 60000, T=cfg.time, label_map=label_map_build,
    move_net=True, batch_size=128,
    encoder_rate_boost=5.5,          # было 3.0 → 1.5
    return_debug=True, desc="Collect train: boost=1.5"
)
Xte, yte, _ = collect_counts_plus_fast(
    net, lif_layer, enc, ds_test, 10000, T=cfg.time, label_map=label_map_build,
    move_net=True, batch_size=128,
    encoder_rate_boost=5.5,
    return_debug=True, desc="Collect test: boost=1.5"
)

N = lif_layer.n
print("mean sum per sample:", float(Xtr[:, :N].sum(1).mean()), float(Xte[:, :N].sum(1).mean()))

In [11]:
N = lif_layer.n
Xtr_n, Xte_n, *_ = tfidf_from_counts(Xtr[:, :N], Xte[:, :N])
_, acc = train_mlp_readout(Xtr_n, ytr, Xte_n, yte, hidden=256, epochs=30, batch_size=256)
print(f"[MLP/TF-IDF] acc = {acc:.3f}")

[MLP/TF-IDF] acc = 0.544


In [7]:
print(dbg_tr)

{'requested_offset': 0.0, 'threshold_abs': None, 'encoder_rate_boost': 3.0, 'temp_disable_inh': False, 'applied_count': 0, 'avg_delta_mean_vt': 0.0, 'restore_ok_fraction': 1.0, 'vt_before_samples': [], 'vt_after_apply_samples': [], 'vt_after_restore_samples': [], 'device': 'cuda', 'batch_size': 64, 'write_pos': 60000, 'expected_batches': 938, 'actual_batches': 938, 'version': 3}


In [ ]:
print("train offset debug:", dbg_tr)

In [ ]:
print(f"[MLP / TF-IDF(counts)] test acc = {acc:.3f}")

In [ ]:
from label_map import load_label_map 
label_map_build = load_label_map ("out/label_map_v5.pt")

In [ ]:
import os
net, input_layer, lif_layer, connection, recurrent_inh, _ = build_net(cfg)
load_weights_into(net, connection, lif_layer, "out/snn_mnist_v5.pt")

In [16]:
from evaluation import eval_readouts_from_net
enc = build_encoder_from_cfg(cfg)
accs = eval_readouts_from_net(net, lif_layer, enc, cfg, label_map=label_map_build)
print(accs)   # ждём, что хотя бы counts_zscore+Linear или PCAwhiten+Linear > 0.6

[collect_counts_plus_cuda] offset summary: requested=+0.000, applied=0, avgΔvt=0.000, restore_ok=–


[collect_counts_plus_cuda] offset summary: requested=+0.000, applied=0, avgΔvt=0.000, restore_ok=–
{'counts_zscore+Linear': 0.4345000088214874, 'PCAwhiten+Linear': 0.42890000343322754, 'TFIDF+MLP': 0.5467000007629395}


In [13]:
# === TRAIN 2 NETS: базовые параметры берём из snn_mnist_v5.pt ===
import os, json
from copy import deepcopy
from pathlib import Path
import torch

from counts_readout import make_mnist_datasets
from snn_mnist_net import run_experiment, save_snn
from label_map import build_label_map, save_label_map

os.makedirs("out", exist_ok=True)

# 1) путь к "последней хорошей сети"
PT_PATH = "snn_mnist_v5.pt"  # <-- если лежит рядом
if not Path(PT_PATH).exists():
    # на всякий случай: если файл в out/
    if Path("out")/PT_PATH:
        PT_PATH = str(Path("out")/PT_PATH)

ckpt = torch.load(PT_PATH, map_location="cpu")
assert isinstance(ckpt, dict) and "cfg" in ckpt, f"В {PT_PATH} нет cfg. keys={list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt)}"
base_cfg_dict = ckpt["cfg"]
assert isinstance(base_cfg_dict, dict), f"ckpt['cfg'] должен быть dict, а не {type(base_cfg_dict)}"

print("Loaded base cfg from:", PT_PATH)
print("Base cfg keys:", list(base_cfg_dict.keys())[:20], "...")

# 2) берём текущий cfg как шаблон (он уже есть в ноутбуке) и накатываем на него базовые поля из pt
cfg_base = deepcopy(cfg)
for k, v in base_cfg_dict.items():
    if hasattr(cfg_base, k):
        setattr(cfg_base, k, v)

# ВАЖНО: базовые параметры одинаковые для всех энкодеров
print(
    f"BASE: seed={cfg_base.seed} device={cfg_base.device} T={cfg_base.time} N={getattr(cfg_base,'N',None)}\n"
    f"      n_hidden={cfg_base.n_hidden} thresh_init={cfg_base.thresh_init}\n"
    f"      STDP: nu_plus={cfg_base.nu_plus} nu_minus={cfg_base.nu_minus}\n"
    f"      FF init: w∈[{cfg_base.w_init_lo},{cfg_base.w_init_hi}], clip=[{cfg_base.w_clip_min},{cfg_base.w_clip_max}]"
)

# 3) датасеты
ds_train, ds_test = make_mnist_datasets()

# 4) какие сети учим (меняется только encoder и его специфические параметры)
RUNS = [
    {
        "name": "poisson",
        "encoder": "poisson",
        "ckpt_path": "out/snn_poisson.pt",
        "label_map_path": "out/label_map_poisson.pt",
    },
    {
        "name": "latency",
        "encoder": "latency",
        "ckpt_path": "out/snn_latency.pt",
        "label_map_path": "out/label_map_latency.pt",
    },
]

meta = {"base_pt": PT_PATH, "base_cfg_dict": base_cfg_dict, "runs": []}

for r in RUNS:
    cfg_run = deepcopy(cfg_base)
    cfg_run.encoder = r["encoder"]   # <-- ключевое различие

    print(f"\n=== TRAIN: {r['name']} / encoder={cfg_run.encoder} ===")
    summary, FF, lif_layer, net, enc = run_experiment(cfg_run, verbose=True, progress=True)
    print("SUMMARY:", summary)

    # сохранить сеть
    save_snn(r["ckpt_path"], cfg_run, FF, lif_layer)
    print("Saved:", r["ckpt_path"])

    # label_map строим отдельно для каждой сети/энкодера
    label_map_build = build_label_map(
        net, None, lif_layer, enc,
        n_calib=2000,
        T=cfg_run.time,
        top_k=getattr(cfg_run, "top_k", 3),
        seed=123
    )
    save_label_map(
        r["label_map_path"],
        label_map_build,
        meta={"ckpt": r["ckpt_path"], "encoder": cfg_run.encoder, "T": cfg_run.time}
    )
    print("Saved label_map:", r["label_map_path"])

    meta["runs"].append({
        "name": r["name"],
        "encoder": cfg_run.encoder,
        "ckpt_path": r["ckpt_path"],
        "label_map_path": r["label_map_path"],
        "summary": summary,
    })

with open("out/train_two_encoders_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("\nDONE. Meta: out/train_two_encoders_meta.json")

Loaded base cfg from: out/snn_mnist_v5.pt
Base cfg keys: ['time', 'n_hidden', 'N', 'seed', 'device', 'log_every', 'debug', 'nu_plus', 'nu_minus', 'enable_inhibition_at_start', 'inhib_strength', 'encoder', 'poisson_rate_scale', 'poisson_base_seed', 'thresh_init', 'thresh_min', 'thresh_max', 'target_spikes', 'ema_alpha', 'ema_k'] ...
BASE: seed=42 device=cpu T=200 N=60000
      n_hidden=100 thresh_init=0.38
      STDP: nu_plus=0.0001 nu_minus=-0.001
      FF init: w∈[0.25,0.8], clip=[0.0,1.0]

=== TRAIN: poisson / encoder=poisson ===
Используем: cpu

SNN-MNIST TRAIN START
seed=42  device=cpu  T=200  N=60000
n_hidden=100  thresh_init=0.38
STDP: nu_plus=0.0001  nu_minus=-0.001
FF init: w∈[0.25, 0.8], clip=[0.0,1.0]
Encoder=poisson  poisson_rate_scale=0.011  base_seed=123
Inhibition: enable=True  inhib_strength=0.705  top_k=3
---------------------------

[lif] thresh = tensor mean=0.380 std=0.000 shape=(100,)
[lif] tc_decay = scalar 150.000
[lif] refrac = scalar 2.000
[lif] reset = scalar 0

Warmup (T=200): 100%|█████████████████| 100/100 [00:12<00:00,  7.71it/s, avg_rate=0.000 target=1.50]
Train (N=60000, T=200):   5%| | 3013/60000 [07:49<2:28:03,  6.41it/s, spikes=0.00 uniq=0 HHI=0.000 e


KeyboardInterrupt: 

In [ ]:
# === Итоговые таблицы для статьи/графиков ===
# Лучший alpha для каждого энкодера (по TF–IDF+MLP) + компромисс по импульсам
metric = "TFIDF+MLP"
cols_show = ["encoder", "alpha", metric, "mean_spikes_test", "counts_zscore+Linear", "PCAwhiten+Linear"]

best_rows = (
    df_sweep.sort_values(metric, ascending=False)
            .groupby("encoder", as_index=False)
            .head(1)
            .loc[:, [c for c in cols_show if c in df_sweep.columns]]
            .sort_values(metric, ascending=False)
)
display(best_rows)

# Таблица «alpha × энкодер» (удобно для построения графиков/теплокарты)
pivot = df_sweep.pivot_table(index="alpha", columns="encoder", values=metric, aggfunc="max")
display(pivot)

# Сохранение краткого отчёта
best_rows.to_csv("out/sweep_best_by_encoder.csv", index=False)
pivot.to_csv("out/sweep_pivot_alpha_by_encoder.csv")
print("Saved: out/sweep_best_by_encoder.csv, out/sweep_pivot_alpha_by_encoder.csv")


In [ ]:
# === Вспомогательные графики (после df_sweep) ===
import matplotlib.pyplot as plt

# Пример 1: точность TF–IDF+MLP от alpha, линии по энкодерам
metric = "TFIDF+MLP"
plt.figure(figsize=(8,5))
for enc_name, sub in df_sweep.groupby("encoder"):
    sub = sub.sort_values("alpha")
    plt.plot(sub["alpha"], sub[metric], marker="o", label=enc_name)
plt.xlabel("alpha (усиление входа)")
plt.ylabel(metric)
plt.title(f"{metric} vs alpha")
plt.grid(True)
plt.legend()
plt.show()

# Пример 2: компромисс «качество–импульсы» (точность vs mean_spikes_test)
plt.figure(figsize=(8,5))
for enc_name, sub in df_sweep.groupby("encoder"):
    plt.plot(sub["mean_spikes_test"], sub[metric], marker="o", label=enc_name)
plt.xlabel("Среднее число импульсов на пример (тест)")
plt.ylabel(metric)
plt.title(f"{metric} vs spikes/sample")
plt.grid(True)
plt.legend()
plt.show()
